# Notebook 24c — Wu 2003 SBI Training: S-B3 pooled multiproposal (eta_col-widened)

**Not executed. Depends on notebook 22d's aggregated bank files.**

Trains `SNPE` on the pooled bank from notebook 22d using the **atomic proposal-correction**
mechanism: each source's rows are appended with their *own* true generating distribution passed as
`proposal=`, and the *declared* prior at `SNPE(prior=...)` construction is the wider target
distribution (`alpha, beta_r, eta_col` uniform; `xi_reb` lognormal). This is the same mechanism this
project has already used elsewhere to pool historically-generated banks drawn from different priors
(see the `scripts/`-based "pooled multiproposal" precedent) -- it is what makes combining notebook
22b's expensive existing bank with a cheaper top-up mathematically valid, instead of a "pretend it's
all i.i.d." hack.

**Why not just retrain from scratch on a fresh full-uniform-on-3-dims bank?** Notebook 22b's mixed-wide
bank took ~14.5 hours to generate. Reusing it (correctly, via proposal tagging) instead of discarding
it is the entire point of this pooling exercise -- notebook 22c's top-up only needs to be big enough
to inform the model about the *additional* `eta_col` coverage, not to re-teach it everything the
existing 5000 rows already cover well (`alpha`, `beta_r`).

In [ ]:
import pathlib, pickle, sys, time, importlib
import numpy as np
import pandas as pd
import torch
from scipy import stats as scipy_stats

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))

from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
import cstr_sbi.recycle.sb3 as sb3
sb3 = importlib.reload(sb3)
from cstr_sbi.recycle.sb3 import SB3_PARAM_NAMES, mixed_wide_prior_4d_sb3, list_closed_loop_sb3, deterministic_window_sb3, classify_fault_sb3
from cstr_sbi.recycle.summaries import compute_summaries, N_SUMMARIES_SB

DATA = ROOT / 'data'
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
FIGURES = ROOT / 'figures'; FIGURES.mkdir(exist_ok=True)
SBI_LOGS = ROOT / 'sbi-logs'; SBI_LOGS.mkdir(exist_ok=True)

POOLED_TRAIN_PATH = DATA / 'wu2003_sbi_train_sb3_pooled_etawide_n7000.npz'
POOLED_SBC_REDUCED_PATH = RESULTS / '24c_sysII_sb3_pooled_etawide_sbc_set_n180.npz'
POOLED_SBC_CONFIRM_PATH = RESULTS / '24c_sysII_sb3_pooled_etawide_sbc_set_n550.npz'

# Held out independently, per source, mirroring this project's dual-independent-SBC-set convention
# (each held-out set is evaluated on its own terms, never pooled into one rank-uniformity test).
OLD_SBC_CONFIRM_PATH = RESULTS / '24b_sysII_sb3_mixedwide_sbc_set_n400.npz'       # kind 0, 2-dim-wide
NEW_SBC_CONFIRM_PATH = RESULTS / '24c_sysII_sb3_etawide_topup_sbc_set_n150.npz'  # kind 1, 3-dim-wide

N_POST = 200
SEEDS = [0, 1, 2, 3, 4]
ARCH, HIDDEN, TRANSFORMS = 'zuko_nsf', 60, 3
MAX_EPOCHS, STOP_AFTER_EPOCHS = 120, 12

print(f'SEEDS={SEEDS}, ARCH={ARCH} {HIDDEN}/{TRANSFORMS}')
print(f'POOLED_TRAIN_PATH={POOLED_TRAIN_PATH.relative_to(ROOT)}')

## Section 1 — Load pooled training data and the two independent held-out SBC sets

In [ ]:
def _load_bank(path, theta_key='thetas4'):
    if not path.exists():
        raise FileNotFoundError(f'{path.name} not found. Run notebook 22d (bank aggregation) first.')
    data = np.load(path)
    return np.asarray(data[theta_key], dtype=np.float32), np.asarray(data['summaries'], dtype=np.float32)

pooled_data = np.load(POOLED_TRAIN_PATH)
theta_train_pooled = np.asarray(pooled_data['thetas4'], dtype=np.float32)
summary_train_pooled = np.asarray(pooled_data['summaries'], dtype=np.float32)
kind_train_pooled = np.asarray(pooled_data['proposal_kind'], dtype=np.int64)
uniform_idx_kind0 = tuple(int(i) for i in pooled_data['uniform_idx_kind0'])
uniform_idx_kind1 = tuple(int(i) for i in pooled_data['uniform_idx_kind1'])

theta_old = theta_train_pooled[kind_train_pooled == 0]
summary_old = summary_train_pooled[kind_train_pooled == 0]
theta_new = theta_train_pooled[kind_train_pooled == 1]
summary_new = summary_train_pooled[kind_train_pooled == 1]
print(f'pooled train: {theta_train_pooled.shape[0]} rows ({theta_old.shape[0]} kind0 uniform_idx={uniform_idx_kind0}, '
      f'{theta_new.shape[0]} kind1 uniform_idx={uniform_idx_kind1})')

theta_sbc_r, summary_sbc_r = _load_bank(POOLED_SBC_REDUCED_PATH)
theta_sbc_c_old, summary_sbc_c_old = _load_bank(OLD_SBC_CONFIRM_PATH)
theta_sbc_c_new, summary_sbc_c_new = _load_bank(NEW_SBC_CONFIRM_PATH)
print(f'sbc_reduced (pooled, screen only): {theta_sbc_r.shape[0]} rows')
print(f'sbc_confirm (old, 2-dim-wide, independent holdout): {theta_sbc_c_old.shape[0]} rows')
print(f'sbc_confirm (new, 3-dim-wide, independent holdout): {theta_sbc_c_new.shape[0]} rows')

# Declared/target prior: the WIDER distribution, matching the top-up bank's uniform_idx.
target_prior = mixed_wide_prior_4d_sb3(uniform_idx=uniform_idx_kind1)
old_proposal = mixed_wide_prior_4d_sb3(uniform_idx=uniform_idx_kind0)
new_proposal = mixed_wide_prior_4d_sb3(uniform_idx=uniform_idx_kind1)  # same object as target_prior here

## Section 2 — Train a seed ensemble via pooled multiproposal `append_simulations`

Each seed's `SNPE` instance is declared with `prior=target_prior` (the wider 3-dim-uniform
distribution), then receives **two** `append_simulations` calls -- one per source, each tagged with
that source's own true generating distribution as `proposal=` -- before a single final `.train()`
call. This is the mechanism that makes pooling mathematically valid rather than a naive
"concatenate and pretend it's i.i.d." shortcut.

In [ ]:
def train_posterior_pooled(seed):
    torch.manual_seed(seed)
    inference = SNPE(prior=target_prior, density_estimator=posterior_nn(model=ARCH, hidden_features=HIDDEN, num_transforms=TRANSFORMS))
    inference.append_simulations(torch.tensor(theta_old), torch.tensor(summary_old), proposal=old_proposal)
    inference.append_simulations(torch.tensor(theta_new), torch.tensor(summary_new), proposal=new_proposal)
    t0 = time.time()
    trained = inference.train(max_num_epochs=MAX_EPOCHS, validation_fraction=0.1,
                              stop_after_epochs=STOP_AFTER_EPOCHS, show_train_summary=False)
    return inference.build_posterior(trained), time.time() - t0


def run_sbc_sb3(posterior, theta_set, summary_set, n_post, seed, label):
    torch.manual_seed(seed)
    ranks = {name: [] for name in SB3_PARAM_NAMES}
    hits90 = {name: 0 for name in SB3_PARAM_NAMES}
    for th, s in zip(theta_set, summary_set):
        with torch.no_grad():
            samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
                                    show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
        for k, name in enumerate(SB3_PARAM_NAMES):
            ranks[name].append(int(np.sum(samp[:, k] < th[k])))
            lo, hi = np.percentile(samp[:, k], [5, 95])
            if lo <= th[k] <= hi:
                hits90[name] += 1
    out = {'n_sbc': int(len(theta_set)), 'n_post': int(n_post)}
    for name in SB3_PARAM_NAMES:
        r = np.asarray(ranks[name], dtype=float)
        ks = scipy_stats.ks_1samp(r / n_post, scipy_stats.uniform.cdf)
        out[name] = {'ks_pvalue': float(ks.pvalue), 'mean_rank_frac': float(r.mean() / n_post),
                     'coverage_90': float(hits90[name] / len(theta_set))}
    out['min_ks_pvalue'] = min(out[name]['ks_pvalue'] for name in SB3_PARAM_NAMES)
    return out


def flatten(seed, label, result):
    row = {'seed': seed, 'holdout': label, 'min_ks_pvalue': result['min_ks_pvalue']}
    for name in SB3_PARAM_NAMES:
        row[f'{name}_ks_pvalue'] = result[name]['ks_pvalue']
    return row


rows = []
posteriors = {}
for seed in SEEDS:
    posterior_path = SBI_LOGS / f'wu2003_posterior_sb3_pooled_etawide_seed{seed}.pkl'
    if posterior_path.exists():
        with open(posterior_path, 'rb') as f:
            posterior = pickle.load(f)['posterior']
        print(f'loaded {posterior_path.name}')
    else:
        posterior, train_wall_s = train_posterior_pooled(seed)
        with open(posterior_path, 'wb') as f:
            pickle.dump({'posterior': posterior, 'train_wall_s': train_wall_s, 'seed': seed,
                         'structure': 'S-B3', 'bank': str(POOLED_TRAIN_PATH)}, f)
        print(f'trained + saved {posterior_path.name} in {train_wall_s:.1f}s')
    posteriors[seed] = posterior
    res_old = run_sbc_sb3(posterior, theta_sbc_c_old, summary_sbc_c_old, n_post=N_POST, seed=248000 + seed, label='old_confirm')
    res_new = run_sbc_sb3(posterior, theta_sbc_c_new, summary_sbc_c_new, n_post=N_POST, seed=249000 + seed, label='new_confirm')
    rows.append(flatten(seed, 'old_2dimwide_n400', res_old))
    rows.append(flatten(seed, 'new_3dimwide_n150', res_new))
    print(f'  seed {seed}: old_confirm min_ks={res_old["min_ks_pvalue"]:.4g}, new_confirm min_ks={res_new["min_ks_pvalue"]:.4g}')

sbc_df = pd.DataFrame(rows)
sbc_df.to_csv(RESULTS / '24c_pooled_etawide_confirm_sbc.csv', index=False)

## Section 3 — Select best seed and save production posterior

Selection is based on the **new** (3-dim-wide) confirm set primarily, since that is the target
distribution this whole exercise is meant to calibrate against; the old confirm set is reported for
cross-check (it should stay well-calibrated too, since `alpha`/`beta_r` coverage did not change).

In [ ]:
new_scores = sbc_df[sbc_df['holdout'] == 'new_3dimwide_n150'].set_index('seed')
old_scores = sbc_df[sbc_df['holdout'] == 'old_2dimwide_n400'].set_index('seed')
print('=== Confirm-budget SBC by seed (new/eta_col-widened holdout, the target check) ===')
print(new_scores[['min_ks_pvalue'] + [f'{n}_ks_pvalue' for n in SB3_PARAM_NAMES]].to_string())
print()
print('=== Confirm-budget SBC by seed (old/2-dim-wide holdout, cross-check) ===')
print(old_scores[['min_ks_pvalue'] + [f'{n}_ks_pvalue' for n in SB3_PARAM_NAMES]].to_string())

best_seed = int(new_scores['min_ks_pvalue'].idxmax())
best_posterior = posteriors[best_seed]
print(f'\nSelected seed {best_seed} by new-holdout min KS p-value = {new_scores.loc[best_seed, "min_ks_pvalue"]:.4g}')
print(f'Same seed on old holdout: min KS p-value = {old_scores.loc[best_seed, "min_ks_pvalue"]:.4g}')

with open(SBI_LOGS / 'wu2003_posterior_sb3_pooled_etawide.pkl', 'wb') as f:
    pickle.dump({
        'posterior': best_posterior, 'selected_seed': best_seed, 'structure': 'S-B3',
        'bank': str(POOLED_TRAIN_PATH), 'sbc_confirm_old': str(OLD_SBC_CONFIRM_PATH),
        'sbc_confirm_new': str(NEW_SBC_CONFIRM_PATH), 'arch': ARCH, 'hidden_features': HIDDEN,
        'num_transforms': TRANSFORMS, 'uniform_idx': uniform_idx_kind1,
        'method': 'pooled_multiproposal (kind0 uniform_idx=(0,1) + kind1 uniform_idx=(0,1,2))',
    }, f)
print('saved sbi-logs/wu2003_posterior_sb3_pooled_etawide.pkl')

## Final acceptance criteria

This notebook passes if, for the selected seed, evaluated against the **new** (eta_col-widened)
confirm holdout:
1. at least `alpha`, `beta_r`, `eta_col` clear `KS p > 0.05` (the three widened dimensions),
2. `min_ks_pvalue` across all four parameters improves on notebook 24b's original result (7.3e-05),
3. the old (2-dim-wide) holdout's `alpha`/`beta_r` calibration is not degraded by the pooling
   (still `KS p > 0.05`) -- i.e. adding the top-up did not break what already worked.

In [ ]:
new_row = new_scores.loc[best_seed]
old_row = old_scores.loc[best_seed]
criteria = [
    ('alpha KS p > 0.05 (new holdout)', float(new_row['alpha_ks_pvalue']) > 0.05, float(new_row['alpha_ks_pvalue'])),
    ('beta_r KS p > 0.05 (new holdout)', float(new_row['beta_r_ks_pvalue']) > 0.05, float(new_row['beta_r_ks_pvalue'])),
    ('eta_col KS p > 0.05 (new holdout)', float(new_row['eta_col_ks_pvalue']) > 0.05, float(new_row['eta_col_ks_pvalue'])),
    ('min_ks_pvalue improves on notebook 24b (7.3e-05)', float(new_row['min_ks_pvalue']) > 7.3e-05, float(new_row['min_ks_pvalue'])),
    ('old holdout alpha/beta_r still calibrated', bool(old_row['alpha_ks_pvalue'] > 0.05 and old_row['beta_r_ks_pvalue'] > 0.05), (float(old_row['alpha_ks_pvalue']), float(old_row['beta_r_ks_pvalue']))),
]
for name, passed, value in criteria:
    print(f"{'PASS' if passed else 'FAIL'} | {name}: {value}")